# Esta parte do código se refere à pipeline da camada BRONZE em BATCH para testes antes de subir ao AWS

In [7]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# basedosdados se refere a base que o Governo Brasileiro disponíbiliza para análises
# pyarrow para salvar em PARQUET

!pip install basedosdados pyarrow --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import basedosdados as bd
import pandas as pd
import hashlib
import logging
import time
import os
from pathlib import Path

from datetime import datetime, timezone

In [9]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
PROJECT_ID = "tech-challenge-fase-2-502101"

DATASET = "br_inep_avaliacao_alfabetizacao"

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos",
    "pib_municipio",
    "indicadores_educacionais_municipio",
    "populacao_municipio",
    "inse_escola"
]

# As 3 últimas tabelas (enriquecimento externo, Fase 3) vêm de datasets
# BigQuery diferentes do dataset principal da Fase 2 - preciso mapear
# qual dataset cada tabela realmente veio, para o metadado
# `_source_dataset` (rastreabilidade) ficar correto em cada uma.
DATASET_POR_TABELA = {
    "pib_municipio": "br_ibge_pib",
    "indicadores_educacionais_municipio": "br_inep_indicadores_educacionais",
    "populacao_municipio": "br_ibge_populacao",
    "inse_escola": "br_inep_indicador_nivel_socioeconomico",
}

BUCKET = "fiap-alfabetizacao-ana-707472259268-us-east-1-an"

INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

CAMADA_BRONZE = "bronze"

In [10]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)

log = logging.getLogger(__name__)

In [11]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA BRONZE")
log.info(f"Projeto GCP : {PROJECT_ID}")
log.info(f"Dataset     : {DATASET}")
log.info(f"Bucket S3   : {BUCKET}")
log.info("~" * 35)

2026-08-24T21:33:02Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-24T21:33:02Z | INFO     | INICIANDO ETL DA CAMADA BRONZE
2026-08-24T21:33:02Z | INFO     | Projeto GCP : tech-challenge-fase-2-502101
2026-08-24T21:33:02Z | INFO     | Dataset     : br_inep_avaliacao_alfabetizacao
2026-08-24T21:33:02Z | INFO     | Bucket S3   : fiap-alfabetizacao-ana-707472259268-us-east-1-an
2026-08-24T21:33:02Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [12]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# NOTA SOBRE FILTRO DE ANO nas tabelas de enriquecimento externo
# (pib_municipio, indicadores_educacionais_municipio, populacao_municipio, inse_escola)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Essas 3 fontes têm histórico desde 1991 no BigQuery. Diferente das 6
# tabelas originais da Fase 2 (que trazem todo o histórico disponível),
# aqui filtramos por WHERE ano IN (...) na extração, por dois motivos:
#   1. O modelo da Fase 3 é uma classificação TRANSVERSAL (não uma
#      previsão temporal): treino e teste usando o MESMO ano-base, 2023
#      -- não "treina em 2023 pra prever 2024". Por isso as 4 fontes de
#      enriquecimento usam só 2023 (não todo o histórico desde 1991, que
#      só infla custo de BigQuery e volume no S3 sem nenhum uso).
#   2. indicadores_educacionais_municipio inclui também 2022, de propósito:
#      é o "ano-1" necessário para o defasamento de 1 ano aplicado na
#      Gold (ver COLUNAS_INDICADORES_DEFASADAS no notebook Gold), que
#      evita usar taxa de aprovação/reprovação do MESMO ano como feature
#      (isso seria data leakage - é resultado do mesmo processo que
#      queremos prever, só que agregado por município).
#
# Se no futuro o projeto evoluir para também aplicar o modelo já treinado
# em uma safra nova de alunos (ex.: 2024), isso é uma etapa de INFERÊNCIA
# sobre o modelo existente - não exige reprocessar Bronze/Silver/Gold
# com um novo ano-base.
#
# IMPORTANTE: isso não contradiz o princípio de "Bronze preserva
# histórico completo" - aquele princípio é sobre não SOBRESCREVER cargas
# já feitas (resolvido via particionamento por ingestion_date). Aqui é
# uma decisão diferente: até onde no tempo faz sentido EXTRAIR da fonte,
# dado o escopo do projeto.

QUERIES = {

    "uf": """
    WITH
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'uf'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'uf'
)
SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
    """,
    "meta_alfabetizacao_brasil": """ 
    SELECT
    dados.ano as ano,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil` AS dados
    """,
     "meta_alfabetizacao_uf": """ 
    SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.rede as rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
    dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
    dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
    dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
    dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
    dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
    dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
    dados.percentual_participacao as percentual_participacao
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
    """,

    "meta_alfabetizacao_municipio": """ 
      SELECT
      dados.ano as ano,
      dados.id_municipio AS id_municipio,
      diretorio_id_municipio.nome AS id_municipio_nome,
      dados.rede as rede,
      dados.taxa_alfabetizacao as taxa_alfabetizacao,
      dados.meta_alfabetizacao_2024 as meta_alfabetizacao_2024,
      dados.meta_alfabetizacao_2025 as meta_alfabetizacao_2025,
      dados.meta_alfabetizacao_2026 as meta_alfabetizacao_2026,
      dados.meta_alfabetizacao_2027 as meta_alfabetizacao_2027,
      dados.meta_alfabetizacao_2028 as meta_alfabetizacao_2028,
      dados.meta_alfabetizacao_2029 as meta_alfabetizacao_2029,
      dados.meta_alfabetizacao_2030 as meta_alfabetizacao_2030,
      dados.nivel_alfabetizacao as nivel_alfabetizacao,
      dados.percentual_participacao as percentual_participacao
  FROM `basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio` AS dados
  LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
      ON dados.id_municipio = diretorio_id_municipio.id_municipio
    """,

    "municipio": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'municipio'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'municipio'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
     """,

    "alunos": """ 
    WITH 
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'alunos'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'alunos'
),
dicionario_presenca AS (
    SELECT
        chave AS chave_presenca,
        valor AS descricao_presenca
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'presenca'
        AND id_tabela = 'alunos'
),
dicionario_preenchimento_caderno AS (
    SELECT
        chave AS chave_preenchimento_caderno,
        valor AS descricao_preenchimento_caderno
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'preenchimento_caderno'
        AND id_tabela = 'alunos'
),
dicionario_alfabetizado AS (
    SELECT
        chave AS chave_alfabetizado,
        valor AS descricao_alfabetizado
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'alfabetizado'
        AND id_tabela = 'alunos'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.id_escola as id_escola,
    dados.id_aluno as id_aluno,
    dados.caderno as caderno,
    descricao_serie AS serie,
    descricao_rede AS rede,
    descricao_presenca AS presenca,
    descricao_preenchimento_caderno AS preenchimento_caderno,
    descricao_alfabetizado AS alfabetizado,
    dados.proficiencia as proficiencia,
    dados.peso_aluno as peso_aluno
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.alunos` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
LEFT JOIN `dicionario_presenca`
    ON dados.presenca = chave_presenca
LEFT JOIN `dicionario_preenchimento_caderno`
    ON dados.preenchimento_caderno = chave_preenchimento_caderno
LEFT JOIN `dicionario_alfabetizado`
    ON dados.alfabetizado = chave_alfabetizado
     """,

    "pib_municipio": """
SELECT
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.ano as ano,
    dados.pib as pib,
    dados.impostos_liquidos as impostos_liquidos,
    dados.va as va,
    dados.va_agropecuaria as va_agropecuaria,
    dados.va_industria as va_industria,
    dados.va_servicos as va_servicos,
    dados.va_adespss as va_adespss
FROM `basedosdados.br_ibge_pib.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio, nome FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE dados.ano = 2023
""",

    "indicadores_educacionais_municipio": """
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.localizacao as localizacao,
    dados.rede as rede,
    dados.atu_ef_anos_iniciais as atu_ef_anos_iniciais,
    dados.had_ef_anos_iniciais as had_ef_anos_iniciais,
    dados.tdi_ef_2_ano as tdi_ef_2_ano,
    dados.taxa_aprovacao_ef_2_ano as taxa_aprovacao_ef_2_ano,
    dados.taxa_reprovacao_ef_2_ano as taxa_reprovacao_ef_2_ano,
    dados.taxa_abandono_ef_2_ano as taxa_abandono_ef_2_ano,
    dados.dsu_ef_anos_iniciais as dsu_ef_anos_iniciais,
    dados.afd_ef_anos_iniciais_grupo_1 as afd_ef_anos_iniciais_grupo_1,
    dados.ird_alta as ird_alta,
    dados.ird_baixa_regularidade as ird_baixa_regularidade,
    dados.icg_nivel_1 as icg_nivel_1,
    dados.icg_nivel_2 as icg_nivel_2,
    dados.icg_nivel_3 as icg_nivel_3,
    dados.icg_nivel_4 as icg_nivel_4,
    dados.icg_nivel_5 as icg_nivel_5,
    dados.icg_nivel_6 as icg_nivel_6
FROM `basedosdados.br_inep_indicadores_educacionais.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio, nome FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE
    dados.ano IN (2022, 2023)
    -- a tabela tem múltiplas linhas por (ano, id_municipio): uma por
    -- combinação de localizacao (Urbana/Rural/Total) x rede
    -- (Estadual/Municipal/Federal/Privada/Pública/Total). Sem esse
    -- filtro, (ano, id_municipio) NÃO é chave única - descoberto ao
    -- rodar com dado real (seria o mesmo bug de chave de negócio
    -- incompleta já corrigido em uf/municipio na Fase 2).
    --
    -- IMPORTANTE: a fonte muda a CAPITALIZAÇÃO entre anos - 2021/2022
    -- usam "total"/"total" (minúsculo), 2023 usa "Total"/"Total"
    -- (maiúsculo, com acento em "Pública"). Descoberto rodando o
    -- diagnóstico real: o filtro original (case-sensitive) só batia
    -- com 2023, fazendo 2022 desaparecer silenciosamente (0 linhas),
    -- o que zerava 100% do defasamento de 1 ano na Gold. Comparação
    -- agora é case-insensitive via LOWER() para cobrir os dois formatos.
    AND LOWER(dados.localizacao) = 'total'
    AND LOWER(dados.rede) = 'total'
""",

    "populacao_municipio": """
SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.populacao as populacao
FROM `basedosdados.br_ibge_populacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT sigla, nome FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio, nome FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE dados.ano IN (2023)
""",

    "inse_escola": """
WITH
dicionario_classificacao AS (
    SELECT
        chave AS chave_classificacao,
        valor AS descricao_classificacao
    FROM `basedosdados.br_inep_indicador_nivel_socioeconomico.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'classificacao'
        AND id_tabela = 'escola'
)
SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    dados.id_escola AS id_escola,
    dados.inse as inse,
    descricao_classificacao AS classificacao
FROM `basedosdados.br_inep_indicador_nivel_socioeconomico.escola` AS dados
LEFT JOIN `dicionario_classificacao`
    ON dados.classificacao = chave_classificacao
WHERE dados.ano IN (2023)
"""

}

In [13]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE LEITURA DAS QUERIES
"""
    Executa uma consulta SQL na Base dos Dados utilizando o BigQuery.

    Args:
        query (str): Consulta SQL a ser executada.

    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_base_dados(query):

    df = bd.read_sql(
        query=query,
        billing_project_id=PROJECT_ID
    )

    return df

In [14]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSTRUINDO A CAMADA BRONZE
"""
    Adiciona metadados de ingestão ao DataFrame da camada Bronze.

    Args:
        df (pandas.DataFrame): Dados originais.
        dataset (str): Nome do dataset de origem.
        tabela (str): Nome da tabela de origem.
        
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def construir_bronze(df, dataset, tabela):

    log.info("Adicionando metadados da camada Bronze")

    df = df.copy()

    df["_ingestion_timestamp"] = INGESTION_TS
    df["_ingestion_date"] = INGESTION_DATE
    df["_source_dataset"] = dataset
    df["_source_table"] = tabela

    df["_record_hash"] = (
        df.astype(str)
          .apply(lambda row: hashlib.md5("".join(row).encode()).hexdigest(), axis=1)
    )

    log.info(f"{len(df)} registros preparados para camada Bronze")

    return df

In [15]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE QUALIDADE
# ~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {  
    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_brasil": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "meta_alfabetizacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio_nome", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
    ],

    "alunos": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_aluno", "critico": True},
        {"tipo": "unique", "coluna": "id_aluno", "critico": False},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "serie", "critico": True},
        {"tipo": "not_null", "coluna": "rede", "critico": True},
        {"tipo": "not_null", "coluna": "presenca", "critico": True},
    ],

    "pib_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "pib", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "indicadores_educacionais_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "populacao_municipio": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "not_null", "coluna": "populacao", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_municipio"], "critico": False},
    ],

    "inse_escola": [
        # min_count crítico: se o SAEB/Prova Brasil não tiver edição em
        # 2023 OU 2024, essa query pode legitimamente vir vazia - queremos
        # que isso pare o pipeline com um erro claro, não passe em
        # silêncio com a tabela vazia.
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "id_escola", "critico": True},
        {"tipo": "not_null", "coluna": "id_municipio", "critico": True},
        {"tipo": "unique", "coluna": ["ano", "id_escola"], "critico": False},
    ]
}

In [16]:
# ~~~~~~~~~~~~~~
# DATA QUALITY
# ~~~~~~~~~~~~~~

def checar_qualidade(df, checks):
    """
    Executa as validações de qualidade da camada Bronze.

    Suporta os tipos: min_count, not_null e unique.
    Respeita o campo `critico`: se True (padrão), uma falha interrompe
    o pipeline (raise Exception); se False, a falha vira apenas um aviso
    no log (WARN) e a execução continua.

    Args:
        df (pandas.DataFrame): DataFrame da Bronze.
        checks (list): Lista de regras de validação.

    Raises:
        Exception: Caso alguma validação com critico=True falhe.
    """

    log.info("Iniciando verificações de qualidade")

    passou = 0
    falhou = 0

    for check in checks:

        tipo = check["tipo"]
        coluna = check.get("coluna")
        valor = check.get("valor")
        critico = check.get("critico", True)

        ok = False
        detalhe = ""

        if tipo == "min_count":

            quantidade = len(df)
            ok = quantidade >= valor
            detalhe = f"quantidade de registros={quantidade} | mínimo esperado={valor}"

        elif tipo == "not_null":

            nulos = df[coluna].isnull().sum()
            ok = nulos == 0
            detalhe = f"coluna '{coluna}' possui {nulos} valor(es) nulo(s)"

        elif tipo == "unique":

            duplicados = df.duplicated(subset=coluna).sum()
            ok = duplicados == 0
            detalhe = f"coluna(s) '{coluna}' possui(em) {duplicados} registro(s) duplicado(s)"

        else:

            log.warning(f"Tipo de check desconhecido, ignorado: '{tipo}'")
            continue

        status = "PASS" if ok else ("FAIL" if critico else "WARN")

        if ok:

            passou += 1
            log.info(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

        else:

            falhou += 1

            if critico:
                log.error(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")
                raise Exception(f"Falha crítica de qualidade ({tipo}): {detalhe}")
            else:
                log.warning(f"[DQ:BRONZE] {status} | {tipo} | {detalhe}")

    log.info(f"Verificações de qualidade concluídas: {passou} passou(aram), {falhou} falhou(aram)")

In [17]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Logging por si só não é observabilidade: para virar métrica consultável
# (ex.: no CloudWatch Logs Insights), o evento precisa ter uma estrutura
# previsível de campo=valor, em vez de texto livre solto na linha de log.

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"

    Args:
        evento (str): Nome do evento (ex.: "tabela_processada", "pipeline_concluido").
        **campos: Pares chave=valor com os dados da métrica (volume, latência etc.).
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro. Sempre loga em nível ERROR (visível em
    qualquer alarme de métrica de erro configurado sobre os logs do
    CloudWatch). Se houver um tópico SNS configurado (variável de
    ambiente SNS_TOPIC_ARN), também publica uma notificação -- para que
    a falha não dependa de alguém abrir o log manualmente (ex.: uma
    carga que falha num sábado).

    Args:
        mensagem (str): Descrição do alerta.
        **contexto: Dados adicionais para diagnóstico (tabela, etapa, erro etc.).
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [18]:
# ~~~~~~~~~~~~~~~~~~~~
# SALVAR CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~

def salvar_bronze(df, tabela):

    # Particiona por ingestion_date (padrão Hive: chave=valor), para que
    # cada execução crie uma pasta nova em vez de sobrescrever a anterior.
    # Isso preserva o histórico completo de cargas, como o README promete,
    # e já deixa o layout pronto para um Glue Crawler detectar partições
    # automaticamente (particionamento físico em Parquet).
    pasta = Path("bronze") / tabela / f"ingestion_date={INGESTION_DATE}"
    pasta.mkdir(parents=True, exist_ok=True)

    arquivo = pasta / f"{tabela}.parquet"

    log.info(f"Salvando arquivo: {arquivo}")

    df.to_parquet(
        arquivo,
        index=False,
        engine="pyarrow"
    )

    log.info("Arquivo Parquet criado com sucesso.")

    return arquivo

In [19]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO PARA CAMADA BRONZE PARA REUTILIZAR EM VÁRIAS TABELAS
"""
    Executa o pipeline completo da camada Bronze para todas as tabelas
    configuradas no dicionário QUERIES.

    Etapas:
        1. Leitura da Base dos Dados.
        2. Construção da camada Bronze.
        3. Validação de qualidade dos dados.
        4. Geração do arquivo Parquet.

    Cada tabela é processada de forma isolada: se uma tabela falhar, um
    alerta é emitido e as demais continuam sendo processadas (a falha de
    uma tabela não derruba o pipeline inteiro). Latência e volume de cada
    tabela são registrados como métricas estruturadas e consultáveis.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_bronze():

    inicio_pipeline = time.perf_counter()

    tabelas_ok = 0
    tabelas_falha = 0

    for tabela, query in QUERIES.items():

        log.info("~" * 35)
        log.info(f"Iniciando execução da camada Bronze para '{tabela}'")
        log.info("~" * 35)

        inicio_tabela = time.perf_counter()

        try:

            # leitura da base dos dados
            df = ler_base_dados(query)

            # construindo a bronze com os metadados
            df_bronze = construir_bronze(df, DATASET_POR_TABELA.get(tabela, DATASET), tabela)

            # exibe as primeiras linhas, somente usado no colab
            print(f"\nPrévia da tabela: {tabela}")
            display(df_bronze.head())

            # checks de integridades e tipos
            checks = CHECKS.get(tabela, [])

            # data quality
            if checks:
                checar_qualidade(df_bronze, checks)

            # salvando
            salvar_bronze(df_bronze, tabela)

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="bronze",
                tabela=tabela,
                volume=len(df_bronze),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            tabelas_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_tabela, 2)

            log_metrica(
                "tabela_processada",
                camada="bronze",
                tabela=tabela,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha na ingestão da tabela '{tabela}' na camada Bronze",
                camada="bronze",
                tabela=tabela,
                erro=str(e)
            )

            tabelas_falha += 1

            # isola a falha: segue para a próxima tabela em vez de
            # derrubar o restante do pipeline
            continue

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="bronze",
        tabelas_ok=tabelas_ok,
        tabelas_falha=tabelas_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if tabelas_falha > 0:
        emitir_alerta(
            f"Pipeline Bronze concluído com {tabelas_falha} falha(s) de {tabelas_ok + tabelas_falha} tabela(s)",
            camada="bronze",
            tabelas_falha=tabelas_falha,
            tabelas_ok=tabelas_ok
        )

    log.info("Camada Bronze concluída!")

In [20]:
executar_bronze()

2026-08-24T21:33:02Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-08-24T21:33:02Z | INFO     | Iniciando execução da camada Bronze para 'uf'
2026-08-24T21:33:02Z | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


Downloading: 100%|██████████|

2026-08-24T21:33:04Z | INFO     | Adicionando metadados da camada Bronze
2026-08-24T21:33:04Z | INFO     | 145 registros preparados para camada Bronze




Prévia da tabela: uf


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,...,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260825_003302,2026-08-25,br_inep_avaliacao_alfabetizacao,uf,857f3dad5f20abbd30fe498de4502ecc
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260825_003302,2026-08-25,br_inep_avaliacao_alfabetizacao,uf,237010d9a4b0e1b02ad5b5b5b2780668
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260825_003302,2026-08-25,br_inep_avaliacao_alfabetizacao,uf,209b69fc0721b75ca21d7679f329d1a6
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260825_003302,2026-08-25,br_inep_avaliacao_alfabetizacao,uf,4379eefa6ccb4b0a69c6c2d5ddbfb73f
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260825_003302,2026-08-25,br_inep_avaliacao_alfabetizacao,uf,3606c89f9684feae1a0a3e7d156a5c7c


...

2026-08-24T21:48:16Z | INFO     | Total time taken 12.82 s.
Finished at 2026-08-24 21:48:16.
2026-08-24T21:48:16Z | INFO     | Adicionando metadados da camada Bronze


2026-08-24T21:48:17Z | INFO     | 69756 registros preparados para camada Bronze



Prévia da tabela: inse_escola


,ano,id_municipio,id_escola,inse,classificacao,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,1200302,12004014,3.89,"Neste nível, os estudantes estão entre um e do...",20260825_003302,2026-08-25,br_inep_indicador_nivel_socioeconomico,inse_escola,b055567819850adf5bca3abc00a9e2fd
1,2023,1200336,12000655,3.72,"Neste nível, os estudantes estão entre um e do...",20260825_003302,2026-08-25,br_inep_indicador_nivel_socioeconomico,inse_escola,0a03da936f71fe39bbc02164ad811d27
2,2023,1300508,13241230,3.45,"Neste nível, os estudantes estão entre um e do...",20260825_003302,2026-08-25,br_inep_indicador_nivel_socioeconomico,inse_escola,a9ea017951813c7ce25b38867d0e254b
3,2023,1300631,13015737,3.97,"Neste nível, os estudantes estão entre um e do...",20260825_003302,2026-08-25,br_inep_indicador_nivel_socioeconomico,inse_escola,a93b08922c95ab887144297180edaa55
4,2023,1300805,13049038,3.68,"Neste nível, os estudantes estão entre um e do...",20260825_003302,2026-08-25,br_inep_indicador_nivel_socioeconomico,inse_escola,497d5b5ed4fc869e81eaa06e7d96d1d9


2026-08-24T21:48:17Z | INFO     | Iniciando verificações de qualidade
2026-08-24T21:48:17Z | INFO     | [DQ:BRONZE] PASS | min_count | quantidade de registros=69756 | mínimo esperado=1
2026-08-24T21:48:17Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'ano' possui 0 valor(es) nulo(s)
2026-08-24T21:48:17Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'id_escola' possui 0 valor(es) nulo(s)
2026-08-24T21:48:17Z | INFO     | [DQ:BRONZE] PASS | not_null | coluna 'id_municipio' possui 0 valor(es) nulo(s)
2026-08-24T21:48:17Z | INFO     | [DQ:BRONZE] PASS | unique | coluna(s) '['ano', 'id_escola']' possui(em) 0 registro(s) duplicado(s)
2026-08-24T21:48:17Z | INFO     | Verificações de qualidade concluídas: 5 passou(aram), 0 falhou(aram)
2026-08-24T21:48:17Z | INFO     | Salvando arquivo: bronze\inse_escola\ingestion_date=2026-08-25\inse_escola.parquet
2026-08-24T21:48:17Z | INFO     | Arquivo Parquet criado com sucesso.
2026-08-24T21:48:17Z | INFO     | [METRICA] evento=tabela_process